# 03 – Features sanity-check EDA

Verify the output of `src/generate_features.py`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

features = pd.read_parquet('../data/processed/features.parquet')
matches = pd.read_parquet('../data/processed/matches.parquet')
print('Features shape:', features.shape)

## 1. Shape check

In [ ]:
print(f'Rows: {len(features):,} | Columns: {features.shape[1]}')
print('Column count breakdown:')
print('  t1_form + t2_form:', sum(1 for c in features.columns if 'form_' in c))
print('  t1_bat + t2_bat:', sum(1 for c in features.columns if '_bat_' in c))
print('  t1_bowl + t2_bowl:', sum(1 for c in features.columns if '_bowl_' in c))
print('  matchup:', sum(1 for c in features.columns if c.startswith('h2h_') or c.startswith('t1_venue') or c.startswith('t2_venue') or c.startswith('venue_avg')))
print('  context:', sum(1 for c in features.columns if c in ['season_match_number','days_into_season','day_of_week','is_weekend']))

## 2. Null audit

In [ ]:
nulls = features.isnull().mean().sort_values(ascending=False) * 100
print(nulls[nulls > 0].to_string() if (nulls > 0).any() else 'No nulls found')
print('Any column >5% null?', (nulls > 5).any())

## 3. Early-season zeros check

First match of 2008 vs last match of 2025 — feature means side by side.

In [ ]:
matches_sorted = matches.sort_values('date')
first_2008 = matches_sorted[matches_sorted['season'] == '2007/08'].iloc[0]
last_2025 = matches_sorted[matches_sorted['season'] == '2025'].iloc[-1]

f08 = features[features['match_id'] == first_2008['match_id']].iloc[0].drop('match_id')
l25 = features[features['match_id'] == last_2025['match_id']].iloc[0].drop('match_id')

comparison = pd.DataFrame({'first_2008': f08, 'last_2025': l25})
print('Mean of first_2008 features:', f08.mean())
print('Mean of last_2025 features:', l25.mean())

## 4. Leakage spot-check — SRH 287 match (2024-04-15, match_id 1426268)

Manually recompute SRH's form-win-rate from matches before this date and assert it matches.

In [ ]:
from src.features import team_form_features

srh_match = matches[matches['match_id'] == '1426268'].iloc[0]
team1 = srh_match['team1']
team2 = srh_match['team2']
cutoff = srh_match['date']

manual_t1 = team_form_features(team1, cutoff, matches, pd.read_parquet('../data/processed/balls.parquet'))
row = features[features['match_id'] == '1426268'].iloc[0]

print('Team1:', team1)
print('Manual form_win_rate:', manual_t1['form_win_rate'])
print('Feature t1_form_win_rate:', row['t1_form_win_rate'])
print('Match:', abs(manual_t1['form_win_rate'] - row['t1_form_win_rate']) < 1e-6)

manual_t2 = team_form_features(team2, cutoff, matches, pd.read_parquet('../data/processed/balls.parquet'))
print('\nTeam2:', team2)
print('Manual form_win_rate:', manual_t2['form_win_rate'])
print('Feature t2_form_win_rate:', row['t2_form_win_rate'])
print('Match:', abs(manual_t2['form_win_rate'] - row['t2_form_win_rate']) < 1e-6)

## 5. Distribution sanity — .describe() of 10 representative features

In [ ]:
rep_cols = [
    't1_form_win_rate', 't1_form_avg_run_rate', 't1_form_avg_economy',
    't1_p1_bat_sr', 't1_p1_bowl_economy', 't1_p1_bowl_dot_pct',
    'h2h_t1_win_rate', 'venue_avg_first_innings_score', 'venue_avg_total_sixes',
    'season_match_number'
]
print(features[rep_cols].describe().T[['mean','min','max']].to_string())

## 6. Time-series smoothness — Mumbai Indians form win rate over time

In [ ]:
mi_matches = matches[(matches['team1'] == 'Mumbai Indians') | (matches['team2'] == 'Mumbai Indians')].sort_values('date')
mi_features = features[features['match_id'].isin(mi_matches['match_id'])].copy()
mi_features = mi_features.merge(mi_matches[['match_id','date']], on='match_id', how='left').sort_values('date')

# Use t1_ if Mumbai is team1, else t2_
mi_features['mi_form_win_rate'] = mi_features.apply(
    lambda r: r['t1_form_win_rate'] if mi_matches[mi_matches['match_id']==r['match_id']].iloc[0]['team1']=='Mumbai Indians' else r['t2_form_win_rate'], axis=1
)

plt.figure(figsize=(12,4))
plt.plot(mi_features['date'], mi_features['mi_form_win_rate'], alpha=0.7)
plt.ylim(0,1)
plt.title('Mumbai Indians: form_win_rate over time')
plt.xlabel('Date')
plt.ylabel('Win rate (last 10 matches)')
plt.tight_layout()
plt.show()